# 🛰️ Orbital Anomaly OpenEnv — Training Notebook v2.1

> **You are the last line of defense for a €500M spacecraft.**  
> 400km above Earth. **36 decision windows**. Ground station out of view.

## What this notebook does
1. Loads the environment locally (no HTTP overhead)
2. Evaluates a **baseline** score before any training
3. Trains with **TRL GRPOTrainer + Unsloth** (the stack recommended by hackathon guides)
4. Evaluates **post-training** across all 3 tasks
5. Plots reward curves, behavioral analysis, mission debriefs

**Runtime:** GPU — Runtime → Change runtime type → **T4 GPU**  
**Time:** ~45-60 min for 100 training steps on T4

## Cell 1 — Install (Run First, Once Per Session)

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Unsloth must be installed before transformers for patching to work
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q bitsandbytes
!pip install -q trl>=0.12.0 transformers>=4.45.0 accelerate peft datasets
!pip install -q openenv-core matplotlib numpy pandas

!rm -rf /content/orbital-anomaly-openenv
!git clone -q https://github.com/umed-indulkar/orbital-anomaly-openenv.git

import sys
sys.path.insert(0, '/content/orbital-anomaly-openenv')
print('Done. Restart runtime if this is the first install, then re-run from Cell 2.')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 15.7 MB/

## Cell 2 — Imports

In [2]:
import warnings, gc, json, random
warnings.filterwarnings('ignore')

import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datasets import Dataset
from unsloth import FastLanguageModel

from server.orbital_anomaly_openenv_environment import OrbitalAnomalyOpenenvEnvironment
from models import OrbitalAnomalyOpenenvAction

VALID_ACTIONS = [
    'rotate_to_sun', 'disable_payload', 'reboot_comms',
    'enter_safe_mode', 'switch_power_bus', 'noop'
]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('Imports OK')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
Imports OK


## Cell 3 — Environment Sanity Check

In [3]:
for task in ['easy', 'medium', 'hard']:
    e = OrbitalAnomalyOpenenvEnvironment()
    o = e.reset(task_id=task)
    assert 0 < o.reward < 1, f'Reward out of (0,1): {o.reward}'
    step_o = e.step(OrbitalAnomalyOpenenvAction(action_type='rotate_to_sun'))
    assert 0 < step_o.reward < 1
    print(f'  {task:6s}: reset_r={o.reward:.4f}  '
          f'soc={o.battery_soc:.1f}%  '
          f'temp={o.thermal_temp:.1f}C  '
          f'status={o.mission_status}')

print('All tasks pass reward range check (0, 1)')

  easy  : reset_r=0.4501  soc=38.0%  temp=38.0C  status=stable
  medium: reset_r=0.4501  soc=61.0%  temp=68.0C  status=warning
  hard  : reset_r=0.4501  soc=22.0%  temp=72.0C  status=warning
All tasks pass reward range check (0, 1)


## Cell 4 — Load Model (4-bit + LoRA via Unsloth)

4-bit quantisation + LoRA = only ~4GB VRAM needed, fits T4 with room for training.

In [4]:
MODEL_NAME  = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
LORA_RANK   = 16
MAX_SEQ_LEN = 512

print(f'Loading {MODEL_NAME}...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    target_modules             = ['q_proj', 'v_proj', 'k_proj', 'o_proj',
                                   'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha                 = 32,
    lora_dropout               = 0.0,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state               = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({trainable/total:.2%}) — LoRA only')
print(f'GPU alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Trainable: 18,464,768 / 907,081,216 (2.04%) — LoRA only
GPU alloc: 1.26 GB


## Cell 5 — Prompts, Action Parser, Episode Runner

In [5]:
SYSTEM_PROMPT = (
    'You are an autonomous satellite mission control AI.\n'
    'A spacecraft is failing. Choose exactly ONE action per step.\n\n'
    'Available actions:\n'
    '  rotate_to_sun    - realign solar panels\n'
    '  disable_payload  - cut science payload heat/power\n'
    '  reboot_comms     - restore communication link\n'
    '  enter_safe_mode  - emergency protocol\n'
    '  switch_power_bus - activate backup power bus\n'
    '  noop             - do nothing\n\n'
    'Reply with ONLY the action name. Nothing else.'
)


def make_prompt(obs) -> str:
    """Format observation as a text prompt for the LLM."""
    beliefs = obs.metadata.get('fault_beliefs', {})
    top_faults = ', '.join(
        f'{k}({v:.0%})' for k, v in
        sorted(beliefs.items(), key=lambda x: x[1], reverse=True)[:3]
    ) if beliefs else 'unknown'

    def lvl(v, lo, hi):
        return 'CRITICAL' if v < lo else ('WARNING' if v < hi else 'OK')

    return (
        f'TELEMETRY — Phase {obs.metadata.get("phase", 0)+1}/3\n'
        f'Battery:  {obs.battery_soc:.1f}%   [{lvl(obs.battery_soc, 20, 40)}]\n'
        f'Solar:    {obs.solar_efficiency:.2f}    [{lvl(obs.solar_efficiency*100, 30, 60)}]\n'
        f'Thermal:  {obs.thermal_temp:.1f}C  [{"CRITICAL" if obs.thermal_temp>85 else "WARNING" if obs.thermal_temp>68 else "OK"}]\n'
        f'Comms:    {obs.comms_signal:.2f}    [{lvl(obs.comms_signal*100, 30, 60)}]\n'
        f'Payload:  {"ON" if obs.payload_on else "OFF"}  SafeMode: {"ON" if obs.safe_mode else "OFF"}\n'
        f'Sunlit:   {obs.sunlit}  GS: {obs.ground_station_visible}\n'
        f'Status:   {obs.mission_status.upper()}\n'
        f'TopFaults: {top_faults}\n'
        f'Action:'
    )


def parse_action(text: str) -> str:
    t = text.strip().lower()
    for a in VALID_ACTIONS:
        if a in t:
            return a
    return 'noop'


def apply_chat_template(obs) -> str:
    """Format as chat with system + user turns."""
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': make_prompt(obs)},
    ]
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )


def run_episode(task_id='easy', max_steps=12, temperature=0.6):
    """Run one full episode. Returns (avg_reward, actions_list, step_rewards)."""
    FastLanguageModel.for_inference(model)
    env = OrbitalAnomalyOpenenvEnvironment()
    obs = env.reset(task_id=task_id)

    total, step_rewards, actions = 0.0, [], []

    for _ in range(max_steps):
        if env._check_done():
            break

        prompt_text = apply_chat_template(obs)
        inputs = tokenizer(
            prompt_text, return_tensors='pt',
            truncation=True, max_length=MAX_SEQ_LEN
        ).to(device)

        with torch.no_grad():
            out = model.generate(
                input_ids      = inputs['input_ids'],
                attention_mask = inputs['attention_mask'],
                max_new_tokens = 10,
                temperature    = temperature,
                do_sample      = True,
                top_p          = 0.95,
                pad_token_id   = tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )
        action_str = parse_action(generated)
        obs    = env.step(OrbitalAnomalyOpenenvAction(action_type=action_str))
        reward = float(obs.reward or 0.001)
        total += reward
        step_rewards.append(reward)
        actions.append(action_str)

    avg = total / len(step_rewards) if step_rewards else 0.001
    return avg, actions, step_rewards


# Quick test
print('Testing episode runner...')
r, acts, _ = run_episode(task_id='easy', max_steps=4)
print(f'  avg_reward={r:.3f}  actions={acts}')
unique = len(set(acts))
print(f'  unique actions: {unique}  (good if >= 2)')

Testing episode runner...


Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  avg_reward=0.682  actions=['reboot_comms', 'disable_payload', 'disable_payload', 'reboot_comms']
  unique actions: 2  (good if >= 2)


## Cell 6 — Baseline Evaluation (BEFORE Training)

These are your **BEFORE** numbers. Write them down.

In [6]:
print('Evaluating baseline (before training)...')
baseline = {}

for task in ['easy', 'medium', 'hard']:
    rewards, all_acts = [], []
    for i in range(10):
        r, acts, _ = run_episode(task_id=task, max_steps=12)
        rewards.append(r)
        all_acts.extend(acts)
    baseline[task] = rewards
    print(f'  {task:6s}: mean={np.mean(rewards):.3f}  '
          f'min={min(rewards):.3f}  max={max(rewards):.3f}  '
          f'unique_actions={len(set(all_acts))}/6')

print(f'\nBaseline easy mean: {np.mean(baseline["easy"]):.3f}')
print('Write these down — they are your BEFORE numbers.')

Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating baseline (before training)...


Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  easy  : mean=0.687  min=0.638  max=0.743  unique_actions=4/6


Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  medium: mean=0.764  min=0.683  max=0.815  unique_actions=4/6


Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  hard  : mean=0.118  min=0.108  max=0.129  unique_actions=3/6

Baseline easy mean: 0.687
Write these down — they are your BEFORE numbers.


## Cell 7 — Build GRPO Training Dataset

GRPOTrainer needs a dataset of prompts. For each prompt it samples N completions,
runs the reward function on each, and updates toward higher-reward completions.
This is the correct training method per the Self-serve guide (TRL + Unsloth + GRPO).

In [7]:
def build_grpo_dataset(n_samples=300, task_id='easy'):
    """
    Build a dataset of varied satellite telemetry states.
    Each row is one prompt (one decision step).
    We randomise the state by running a few random steps first.
    """
    records = []
    env = OrbitalAnomalyOpenenvEnvironment()

    for i in range(n_samples):
        obs = env.reset(task_id=task_id)
        # Randomise 0-5 steps to get diverse states
        n_rand = random.randint(0, 5)
        for _ in range(n_rand):
            obs = env.step(OrbitalAnomalyOpenenvAction(
                action_type=random.choice(VALID_ACTIONS)
            ))
            if obs.done:
                obs = env.reset(task_id=task_id)

        prompt_text = apply_chat_template(obs)
        records.append({'prompt': prompt_text})

    return Dataset.from_list(records)


print('Building training dataset...')
train_dataset = build_grpo_dataset(n_samples=300, task_id='easy')
print(f'Dataset: {len(train_dataset)} prompts')
print(f'Sample prompt (first 300 chars):\n{train_dataset[0]["prompt"][:300]}...')

Building training dataset...
Dataset: 300 prompts
Sample prompt (first 300 chars):
<|im_start|>system
You are an autonomous satellite mission control AI.
A spacecraft is failing. Choose exactly ONE action per step.

Available actions:
  rotate_to_sun    - realign solar panels
  disable_payload  - cut science payload heat/power
  reboot_comms     - restore communication link
  ente...


## Cell 8 — GRPO Reward Function

The reward function is called by GRPOTrainer for each sampled completion.
It runs the action in the environment and returns the reward.
Using **3 independent reward signals** as recommended by the Self-serve guide.

In [8]:
def satellite_reward(completions, prompts=None, **kwargs):
    """
    GRPO reward function.
    Called with a batch of completions from the model.
    Returns a list of scalar rewards.

    Three independent reward components (anti-hacking):
      1. environment_reward: run action in env, get actual reward
      2. format_reward: did the model output a valid action name?
      3. brevity_reward: penalise verbose outputs
    """
    rewards = []
    env = OrbitalAnomalyOpenenvEnvironment()

    for completion in completions:
        # Extract the text from the completion dict
        if isinstance(completion, list) and len(completion) > 0:
            text = completion[0].get('content', '') if isinstance(completion[0], dict) else str(completion[0])
        else:
            text = str(completion)

        action = parse_action(text)

        # Component 1: environment reward
        obs = env.reset(task_id='easy')
        obs = env.step(OrbitalAnomalyOpenenvAction(action_type=action))
        env_r = float(obs.reward or 0.001)

        # Run 3 more steps to get a more stable signal
        for _ in range(3):
            if env._check_done():
                break
            obs = env.step(OrbitalAnomalyOpenenvAction(action_type=action))
            env_r += float(obs.reward or 0.001)
        env_r /= 4.0

        # Component 2: format reward (is it a valid named action?)
        format_r = 0.1 if action != 'noop' and action in text.lower() else 0.0

        # Component 3: brevity (penalise outputs > 20 chars)
        brevity_r = 0.05 if len(text.strip()) <= 20 else 0.0

        rewards.append(env_r + format_r + brevity_r)

    return rewards


# Test the reward function
test_completions = [
    [{'content': 'rotate_to_sun'}],
    [{'content': 'noop'}],
    [{'content': 'disable_payload'}],
]
test_rewards = satellite_reward(test_completions)
print('Reward function test:')
for c, r in zip(['rotate_to_sun', 'noop', 'disable_payload'], test_rewards):
    print(f'  {c}: {r:.4f}')
print('Reward function OK')

Reward function test:
  rotate_to_sun: 0.9228
  noop: 0.7039
  disable_payload: 0.8266
Reward function OK


## Cell 9 — GRPO Training (TRL + Unsloth)

This is the **correct** training approach per the hackathon Self-serve guide.
GRPOTrainer samples multiple completions per prompt, scores them via
our reward function, and updates toward higher-scoring completions.

Expected training time: ~45-60 min for 100 steps on T4.

In [9]:
from trl import GRPOConfig, GRPOTrainer

FastLanguageModel.for_training(model)
gc.collect()
torch.cuda.empty_cache()

grpo_config = GRPOConfig(
    # Core training
    output_dir              = '/content/orbital_grpo_output',
    num_train_epochs        = 1,
    max_steps               = 100,          # ~45 min on T4
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,        # effective batch = 4
    learning_rate           = 5e-6,

    # GRPO-specific
    num_generations         = 4,            # sample 4 completions per prompt
    max_prompt_length       = 400,
    max_completion_length   = 12,           # action names are short
    temperature             = 0.8,          # exploration during training
    beta                    = 0.001,        # low KL penalty = more exploration

    # Logging
    logging_steps           = 5,
    save_steps              = 50,
    report_to               = 'none',       # no wandb
    remove_unused_columns   = False,
    dataloader_num_workers  = 0,
)

trainer = GRPOTrainer(
    model             = model,
    processing_class  = tokenizer,
    reward_funcs      = [satellite_reward],
    args              = grpo_config,
    train_dataset     = train_dataset,
)

print('Starting GRPO training...')
print(f'Steps: {grpo_config.max_steps} | Batch: {grpo_config.per_device_train_batch_size} | '
      f'Generations per prompt: {grpo_config.num_generations}')
print(f'GPU before training: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print()

train_result = trainer.train()
print()
print(f'Training complete!')
print(f'Final loss: {train_result.training_loss:.4f}')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting GRPO training...
Steps: 100 | Batch: 1 | Generations per prompt: 4
GPU before training: 1.31 GB



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / satellite_reward / mean,rewards / satellite_reward / std
5,-0.000000,0.849212,0.033492,3.750000,3.200000,4.400000,0.000000,3.750000,3.200000,4.400000,0.000022,0.849212,0.033492
10,-0.000002,0.851051,0.030445,3.550000,3.000000,4.400000,0.000000,3.550000,3.000000,4.400000,0.000972,0.851051,0.030445
15,0.000042,0.856282,0.041257,3.900000,3.000000,4.600000,0.000000,3.900000,3.000000,4.600000,0.042151,0.856282,0.041257
20,0.000172,0.895130,0.036462,3.950000,3.600000,4.200000,0.000000,3.950000,3.600000,4.200000,0.172039,0.895130,0.036462
25,0.000164,0.922800,0.000000,4.000000,4.000000,4.000000,0.000000,4.000000,4.000000,4.000000,0.164413,0.922800,0.000000
30,0.000245,0.922800,0.000000,4.000000,4.000000,4.000000,0.000000,4.000000,4.000000,4.000000,0.245334,0.922800,0.000000
35,0.000226,0.922800,0.000000,4.000000,4.000000,4.000000,0.000000,4.000000,4.000000,4.000000,0.226460,0.922800,0.000000
40,0.000209,0.922800,0.000000,4.000000,4.000000,4.000000,0.000000,4.000000,4.000000,4.000000,0.208900,0.922800,0.000000
45,0.000314,0.922800,0.000000,4.000000,4.000000,4.000000,0.000000,4.000000,4.000000,4.000000,0.313609,0.922800,0.000000
50,0.000223,0.922800,0.000000,4.000000,4.000000,4.000000,0.000000,4.000000,4.000000,4.000000,0.223134,0.922800,0.000000


Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


Training complete!
Final loss: 0.0007


## Cell 10 — Extract Training Reward Curve

In [10]:
# Extract reward values from trainer logs
log_history = trainer.state.log_history
print(f'Total log entries: {len(log_history)}')
print(f'Sample entry: {log_history[0] if log_history else "none"}')

# GRPO logs reward under different key names depending on version
reward_keys = ['reward', 'rewards', 'mean_reward', 'train/reward']
train_steps, train_rewards = [], []

for entry in log_history:
    for key in reward_keys:
        if key in entry:
            train_steps.append(entry.get('step', len(train_steps)))
            train_rewards.append(float(entry[key]))
            break

print(f'Found {len(train_rewards)} reward log entries')
if train_rewards:
    print(f'First reward: {train_rewards[0]:.4f}')
    print(f'Last reward:  {train_rewards[-1]:.4f}')
    print(f'Best reward:  {max(train_rewards):.4f}')
else:
    print('No reward logs found — will use episode eval rewards instead')

Total log entries: 21
Sample entry: {'loss': -7.748603820800782e-08, 'grad_norm': 8.414694786071777, 'learning_rate': 2.0000000000000003e-06, 'num_tokens': 4567.0, 'completions/mean_length': 3.75, 'completions/min_length': 3.2, 'completions/max_length': 4.4, 'completions/clipped_ratio': 0.0, 'completions/mean_terminated_length': 3.75, 'completions/min_terminated_length': 3.2, 'completions/max_terminated_length': 4.4, 'rewards/satellite_reward/mean': 0.8492124915122986, 'rewards/satellite_reward/std': 0.0334923202637583, 'reward': 0.8492124915122986, 'reward_std': 0.03349231951870024, 'frac_reward_zero_std': 0.2, 'completion_length': 3.75, 'kl': 2.2056798218805795e-05, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, 'epoch': 0.016666666666666666, 'step': 5}
Found 20 reward log entries
First reward: 0.8492
Last reward:  0.9228
Best reward:  0.9228


## Cell 11 — Post-Training Evaluation

In [11]:
FastLanguageModel.for_inference(model)

print('Evaluating after training...')
post = {}

for task in ['easy', 'medium', 'hard']:
    rewards, all_acts = [], []
    for i in range(10):
        r, acts, _ = run_episode(task_id=task, max_steps=12)
        rewards.append(r)
        all_acts.extend(acts)
    post[task] = rewards
    print(f'  {task:6s}: mean={np.mean(rewards):.3f}  '
          f'min={min(rewards):.3f}  max={max(rewards):.3f}  '
          f'unique_actions={len(set(all_acts))}/6')

print()
print(f'{"Task":<10} {"Before":>10} {"After":>10} {"Delta":>10} {"Result"}')
print('-' * 52)
for task in ['easy', 'medium', 'hard']:
    pre  = np.mean(baseline[task])
    post_m = np.mean(post[task])
    delta  = post_m - pre
    pct    = delta / pre * 100
    result = 'IMPROVED' if delta > 0.01 else ('SIMILAR' if abs(delta) < 0.01 else 'WORSE')
    print(f'{task:<10} {pre:>10.3f} {post_m:>10.3f} {pct:>+9.1f}%  {result}')

Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating after training...


Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  easy  : mean=0.640  min=0.640  max=0.640  unique_actions=1/6


Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  medium: mean=0.593  min=0.593  max=0.593  unique_actions=1/6


Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  hard  : mean=0.092  min=0.092  max=0.092  unique_actions=1/6

Task           Before      After      Delta Result
----------------------------------------------------
easy            0.687      0.640      -6.8%  WORSE
medium          0.764      0.593     -22.4%  WORSE
hard            0.118      0.092     -22.0%  WORSE


## Cell 12 — Training Results Plot

In [12]:
def smooth(v, w=5):
    return [np.mean(v[max(0,i-w):i+1]) for i in range(len(v))]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0a0e1a')
for ax in axes:
    ax.set_facecolor('#111827')
    for sp in ax.spines.values(): sp.set_color('#374151')
    ax.tick_params(colors='#9ca3af')
    ax.xaxis.label.set_color('#9ca3af')
    ax.yaxis.label.set_color('#9ca3af')
    ax.title.set_color('#f9fafb')

# Panel 1: Training reward curve from GRPO logs
if train_rewards:
    sm = smooth(train_rewards, w=3)
    axes[0].plot(train_steps, train_rewards, color='#374151', alpha=0.4,
                 linewidth=1, label='Raw')
    axes[0].plot(train_steps, sm, color='#3b82f6', linewidth=2.5, label='Smoothed')
    axes[0].axhline(y=np.mean(baseline['easy']), color='#ef4444',
                    linestyle='--', label=f'Baseline: {np.mean(baseline["easy"]):.3f}')
    axes[0].legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
else:
    # Fallback: episode rewards
    ep_r = [np.mean(baseline['easy'])] + [np.mean(post['easy'])]
    axes[0].bar(['Before', 'After'], ep_r, color=['#ef4444', '#10b981'])
axes[0].set_title('GRPO Training Reward Curve (Easy)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Reward')
axes[0].grid(alpha=0.2, color='#374151')

# Panel 2: Before vs After all tasks
task_labels = ['Easy', 'Medium', 'Hard']
pre_means   = [np.mean(baseline[t.lower()]) for t in task_labels]
post_means  = [np.mean(post[t.lower()]) for t in task_labels]
x = np.arange(len(task_labels))
w = 0.35
axes[1].bar(x-w/2, pre_means,  w, color='#ef4444', alpha=0.85, label='Before')
axes[1].bar(x+w/2, post_means, w, color='#10b981', alpha=0.85, label='After')
for i, (p, q) in enumerate(zip(pre_means, post_means)):
    axes[1].text(i-w/2, p+0.005, f'{p:.3f}', ha='center', color='white', fontsize=9)
    axes[1].text(i+w/2, q+0.005, f'{q:.3f}', ha='center', color='white', fontsize=9)
axes[1].set_title('Pre vs Post Training — All Tasks', fontsize=12, fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(task_labels)
axes[1].set_ylabel('Avg Step Reward')
axes[1].legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
axes[1].grid(axis='y', alpha=0.2, color='#374151')

# Panel 3: Action distribution shift
def act_dist(task, rewards_dict):
    _, acts, _ = run_episode(task_id=task, max_steps=24)
    counts = {a: acts.count(a)/len(acts) for a in VALID_ACTIONS}
    return counts

pre_dist  = act_dist('easy', baseline)
post_dist = act_dist('easy', post)
labels = [a.replace('_', '\\n') for a in VALID_ACTIONS]
px = np.arange(len(VALID_ACTIONS))
w2 = 0.35
axes[2].bar(px-w2/2, [pre_dist[a]  for a in VALID_ACTIONS], w2,
            color='#ef4444', alpha=0.85, label='Before')
axes[2].bar(px+w2/2, [post_dist[a] for a in VALID_ACTIONS], w2,
            color='#10b981', alpha=0.85, label='After')
axes[2].set_title('Action Distribution Shift (Easy)', fontsize=11, fontweight='bold')
axes[2].set_xticks(px)
axes[2].set_xticklabels(VALID_ACTIONS, fontsize=7, rotation=20)
axes[2].set_ylabel('Fraction of Steps')
axes[2].legend(facecolor='#1f2937', labelcolor='white', fontsize=9)
axes[2].grid(axis='y', alpha=0.2, color='#374151')

plt.suptitle('Orbital Anomaly OpenEnv — Training Results (GRPO)',
             fontsize=13, color='#f9fafb', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0e1a')
plt.show()
print('Saved training_results.png')

try:
    from google.colab import files
    files.download('training_results.png')
except:
    print('(download skipped — not in Colab)')

Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Saved training_results.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 13 — Behavioral Analysis

What did the model **actually learn**? This is the story judges want to hear.

In [13]:
print('=== BEHAVIORAL ANALYSIS — What Did Training Change? ===')
print()

# Run 3 episodes before/after on hard task to show concrete differences
FastLanguageModel.for_inference(model)

# Compare action sequences on hard task
hard_actions_pre  = []
hard_actions_post = []

# Pre-training proxy: use untrained decisions (from baseline eval)
# Post-training: run model now
for _ in range(3):
    _, acts, _ = run_episode(task_id='hard', max_steps=12)
    hard_actions_post.extend(acts)

print('Post-training hard task actions (3 episodes):')
from collections import Counter
ctr = Counter(hard_actions_post)
for act, cnt in ctr.most_common():
    pct = cnt / len(hard_actions_post) * 100
    bar = '#' * int(pct / 5)
    print(f'  {act:<22} {bar:<20} {pct:.0f}%')

print()
print('Pre vs Post action distribution (easy task):')
print(f'{"Action":<24} {"Before":>8} {"After":>8} {"Change"}')
print('-' * 55)
for act in VALID_ACTIONS:
    pre_p  = pre_dist.get(act, 0)
    post_p = post_dist.get(act, 0)
    delta  = post_p - pre_p
    arrow  = 'UP' if delta > 0.05 else ('DOWN' if delta < -0.05 else '~')
    print(f'{act:<24} {pre_p:>8.1%} {post_p:>8.1%}  {arrow} ({delta:+.1%})')

print()
print('Key insight:')
best_post = max(post_dist, key=post_dist.get)
print(f'  Most-chosen action after training: {best_post}')
print(f'  This reflects what the environment rewarded most in training.')
print()
print('For the pitch, explain:')
print('  "The model learned to prioritize [X] in the early steps because')
print('   it has the highest causal impact on the reward trajectory."')

Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== BEHAVIORAL ANALYSIS — What Did Training Change? ===



Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Post-training hard task actions (3 episodes):
  rotate_to_sun          #################### 100%

Pre vs Post action distribution (easy task):
Action                     Before    After Change
-------------------------------------------------------
rotate_to_sun              100.0%   100.0%  ~ (+0.0%)
disable_payload              0.0%     0.0%  ~ (+0.0%)
reboot_comms                 0.0%     0.0%  ~ (+0.0%)
enter_safe_mode              0.0%     0.0%  ~ (+0.0%)
switch_power_bus             0.0%     0.0%  ~ (+0.0%)
noop                         0.0%     0.0%  ~ (+0.0%)

Key insight:
  Most-chosen action after training: rotate_to_sun
  This reflects what the environment rewarded most in training.

For the pitch, explain:
  "The model learned to prioritize [X] in the early steps because
   it has the highest causal impact on the reward trajectory."


## Cell 14 — Save LoRA Adapters

In [14]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/orbital_anomaly_lora'
except:
    SAVE_PATH = '/content/orbital_anomaly_lora'

os.makedirs(SAVE_PATH, exist_ok=True)

# Save LoRA adapters (~50MB, much smaller than full model)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'LoRA adapters saved to {SAVE_PATH}')

# Save training summary
summary = {
    'model':       MODEL_NAME,
    'lora_rank':   LORA_RANK,
    'grpo_steps':  grpo_config.max_steps,
    'baseline':    {t: float(np.mean(v)) for t, v in baseline.items()},
    'post':        {t: float(np.mean(v)) for t, v in post.items()},
    'improvement': {
        t: f'{(np.mean(post[t])-np.mean(baseline[t]))/np.mean(baseline[t])*100:+.1f}%'
        for t in ['easy','medium','hard']
    },
}
with open(f'{SAVE_PATH}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Summary:')
for task, val in summary['improvement'].items():
    print(f'  {task}: {val}')

Mounted at /content/drive


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/orbital_anomaly_lora/tokenizer_config.json.


LoRA adapters saved to /content/drive/MyDrive/orbital_anomaly_lora
Summary:
  easy: -6.8%
  medium: -22.4%
  hard: -22.0%


## Cell 15 — Final Summary for Pitch

In [15]:
print('=' * 65)
print('ORBITAL ANOMALY OPENENV — TRAINING COMPLETE')
print('=' * 65)
print()
print('Results:')
for task in ['easy', 'medium', 'hard']:
    pre  = np.mean(baseline[task])
    pm   = np.mean(post[task])
    pct  = (pm - pre) / pre * 100
    sym  = 'IMPROVED' if pct > 1 else ('FLAT' if abs(pct) < 1 else 'WORSE')
    print(f'  {task:<8}: {pre:.3f} -> {pm:.3f}  ({pct:+.1f}%)  {sym}')

print()
print('Training method: TRL GRPOTrainer + Unsloth 4-bit LoRA')
print('Theme alignment:')
print('  Theme 3 (World Modeling):  13-fault belief state, causal reasoning')
print('  Theme 2 (Long-Horizon):    36-step Extended Mission Mode')
print('  Theme 1 (Multi-Agent):     Commander + 3 Specialist Agents')
print()
print('Files created:')
print('  training_results.png - download and use in blog post')
print(f'  {SAVE_PATH}/ - LoRA adapters')

ORBITAL ANOMALY OPENENV — TRAINING COMPLETE

Results:
  easy    : 0.687 -> 0.640  (-6.8%)  WORSE
  medium  : 0.764 -> 0.593  (-22.4%)  WORSE
  hard    : 0.118 -> 0.092  (-22.0%)  WORSE

Training method: TRL GRPOTrainer + Unsloth 4-bit LoRA
Theme alignment:
  Theme 3 (World Modeling):  13-fault belief state, causal reasoning
  Theme 2 (Long-Horizon):    36-step Extended Mission Mode
  Theme 1 (Multi-Agent):     Commander + 3 Specialist Agents

Files created:
  training_results.png - download and use in blog post
  /content/drive/MyDrive/orbital_anomaly_lora/ - LoRA adapters
